In [4]:
import os
print(os.getcwd())

/Users/mac/Desktop/semantic_graph_project/notebooks


In [5]:
# Cell 1 — Read and inspect the raw Dutch association data.
# GOAL: confirm the file parses correctly with the right delimiter/quote/encoding,
# and that the columns are what we think, BEFORE any aggregation.

import os
from pathlib import Path
import pandas as pd

# Anchor to project root regardless of where the kernel started.
# The notebook lives in notebooks/, so root is its parent.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Project root: {ROOT}")

NL_RAW = ROOT / "data/raw/DutchWordAssociationData_April2012CSV/associationData.csv"
assert NL_RAW.exists(), f"File not found at {NL_RAW}"

try:
    nl = pd.read_csv(NL_RAW, sep=";", quotechar='"', encoding="utf-8",
                     dtype=str, keep_default_na=False)
    enc_used = "utf-8"
except UnicodeDecodeError:
    nl = pd.read_csv(NL_RAW, sep=";", quotechar='"', encoding="latin-1",
                     dtype=str, keep_default_na=False)
    enc_used = "latin-1"

print(f"Encoding used: {enc_used}")
print(f"Shape: {nl.shape}")
print(f"Columns: {list(nl.columns)}")
print("\nFirst 5 rows:")
print(nl.head())

print(f"\nUnique cues: {nl['cue'].nunique()}")
print(f"Unique participants: {nl['recodedPP_ID'].nunique()}")
print(f"Total trial rows: {len(nl)}")

for col in ["asso1", "asso2", "asso3"]:
    n_empty = (nl[col].str.strip() == "").sum()
    print(f"  {col}: {n_empty:,} empty ({100*n_empty/len(nl):.1f}%)")

Project root: /Users/mac/Desktop/semantic_graph_project
Encoding used: utf-8
Shape: (1257100, 6)
Columns: ['ctr', 'recodedPP_ID', 'cue', 'asso1', 'asso2', 'asso3']

First 5 rows:
      ctr recodedPP_ID       cue      asso1             asso2       asso3
0  272276        16289  oneindig     eeuwig  onoverzichtelijk   eindeloos
1  273869        16368  oneindig        ver              lang  vraagteken
2  273880        16369  oneindig  onbeperkt            heelal  duisternis
3  273912        16367  oneindig       ruim              veel    wiskunde
4  273931        16370  oneindig  eindeloos       ongrijpbaar      heelal

Unique cues: 12571
Unique participants: 70369
Total trial rows: 1257100
  asso1: 0 empty (0.0%)
  asso2: 0 empty (0.0%)
  asso3: 0 empty (0.0%)


In [6]:
# Cell 2 — Hunt for sentinel/placeholder tokens before aggregating.
# WHY: 0.0% empty across all three slots is unusual vs English SWOW, where
# asso3 is often blank. Either this protocol forced 3 responses, or "empty"
# is encoded as a placeholder string that would become a fake hub node.
# We look at the most frequent response tokens; a sentinel would stand out.

import pandas as pd

# Stack all three response columns into one Series to see global token freq.
all_responses = pd.concat([nl["asso1"], nl["asso2"], nl["asso3"]], ignore_index=True)

print(f"Total response tokens: {len(all_responses):,}")
print(f"Unique response tokens: {all_responses.nunique():,}\n")

print("Top 25 most frequent responses across all slots:")
print(all_responses.value_counts().head(25))

# Explicitly probe for common SWOW/Dutch sentinels and odd values.
print("\nExplicit sentinel probes (count of exact matches):")
for token in ["", " ", "NA", "na", "x", "X", "geen", "niets", "nee",
              "onbekend", "?", "-", "null", "Missing", "MISSING"]:
    c = (all_responses == token).sum()
    if c > 0:
        print(f"  {token!r}: {c:,}")

# How many response cells are non-empty but look non-lexical (very short)?
lengths = all_responses.str.len()
print(f"\nResponses of length 1: {(lengths == 1).sum():,}")
print(f"Responses of length 0: {(lengths == 0).sum():,}")

Total response tokens: 3,771,300
Unique response tokens: 201,356

Top 25 most frequent responses across all slots:
x           124561
water        18530
geld         16202
eten         15567
pijn         12356
auto         12333
lekker       11665
muziek       11536
mooi         10446
kinderen      9902
school        9684
warm          9523
zee           9462
rood          9021
groen         8681
kind          8671
wit           7950
werk          7378
vrouw         7305
vies          7286
klein         7280
oud           7113
man           7030
koud          6973
zon           6769
Name: count, dtype: int64

Explicit sentinel probes (count of exact matches):
  'na': 189
  'x': 124,561
  'geen': 535
  'niets': 1,261
  'nee': 536
  'onbekend': 4
  '?': 10
  '-': 5
  'null': 1

Responses of length 1: 125,522
Responses of length 0: 0


In [7]:
# Cell 3 — Manufacture an English-equivalent strength table from raw Dutch trials.
# WHY: English Phase 1 was handed columns (cue, response, R123, N, R123.Strength).
# Dutch gives us raw participant-cue trials. We must rebuild R123 and N with the
# SAME definitions as English, or the edge weights are on a different scale and
# every cross-linguistic weight comparison silently breaks.
#
# English definitions we are matching:
#   R123 = number of DISTINCT PARTICIPANTS who gave that response to that cue
#          (pooled across asso1/asso2/asso3; a participant counted once per response)
#   N    = number of DISTINCT PARTICIPANTS who saw that cue
#   R123.Strength = R123 / N

import pandas as pd
import re

# --- Step A: melt the three response slots into long form -------------------
# Each (participant, cue) trial contributes up to 3 (cue, response) rows.
long = nl.melt(
    id_vars=["recodedPP_ID", "cue"],
    value_vars=["asso1", "asso2", "asso3"],
    value_name="response",
).drop(columns="variable")

# Lowercase cue+response NOW so participant-dedup and counting are case-correct,
# matching English where lowercasing precedes everything.
long["cue"] = long["cue"].str.lower().str.strip()
long["response"] = long["response"].str.lower().str.strip()

# Drop empty responses (defensive — we saw 0 truly-empty, but strip may expose some)
long = long[long["response"] != ""]

print(f"Long-form response rows (pre-dedup): {len(long):,}")

# --- Step B: N = distinct participants per cue ------------------------------
# Computed from the ORIGINAL trials, not the melted responses, so it equals
# "participants who saw this cue" exactly like English N.
N_per_cue = nl.assign(cue=nl["cue"].str.lower().str.strip()) \
              .groupby("cue")["recodedPP_ID"].nunique()
print(f"Cues with an N value: {len(N_per_cue):,}")
print(f"N range: min={N_per_cue.min()}, median={int(N_per_cue.median())}, max={N_per_cue.max()}")

# --- Step C: R123 = distinct participants per (cue, response) ---------------
# Dedup within participant so a word given in two slots by one person counts ONCE.
long_dedup = long.drop_duplicates(subset=["cue", "response", "recodedPP_ID"])
print(f"Long-form rows after within-participant dedup: {len(long_dedup):,}")

R123 = (long_dedup
        .groupby(["cue", "response"])["recodedPP_ID"]
        .size()
        .rename("R123")
        .reset_index())

# --- Step D: assemble the English-shaped strength table ---------------------
nl_strength = R123.merge(N_per_cue.rename("N"), on="cue", how="left")
nl_strength["R123.Strength"] = nl_strength["R123"] / nl_strength["N"]

print(f"\nManufactured strength table shape: {nl_strength.shape}")
print(f"Columns: {list(nl_strength.columns)}")

# --- Inspect a known cue BEFORE any cleaning --------------------------------
# 'water' should have sensible Dutch associations with believable proportions.
print("\n=== 'water' top 10 by strength (raw, pre-cleaning) ===")
print(nl_strength[nl_strength["cue"] == "water"]
      .sort_values("R123.Strength", ascending=False)
      .head(10).to_string(index=False))

# Sanity: strengths must be in (0, 1]; flag anything pathological.
bad = nl_strength[(nl_strength["R123.Strength"] <= 0) | (nl_strength["R123.Strength"] > 1)]
print(f"\nRows with out-of-range strength (should be 0): {len(bad)}")

Long-form response rows (pre-dedup): 3,771,300
Cues with an N value: 12,571
N range: min=98, median=100, max=100
Long-form rows after within-participant dedup: 3,719,198

Manufactured strength table shape: (1497336, 5)
Columns: ['cue', 'response', 'R123', 'N', 'R123.Strength']

=== 'water' top 10 by strength (raw, pre-cleaning) ===
  cue response  R123   N  R123.Strength
water  drinken    31 100           0.31
water    dorst    27 100           0.27
water      zee    27 100           0.27
water  zwemmen    22 100           0.22
water      nat    15 100           0.15
water    blauw    14 100           0.14
water     fles     7 100           0.07
water  zwembad     6 100           0.06
water   rivier     6 100           0.06
water     fris     6 100           0.06

Rows with out-of-range strength (should be 0): 0


In [8]:
# Cell 4 — Apply the EXACT Phase 1 cleaning + aggregation to the Dutch strength
# table, then build the directed graph and report Phase-1-equivalent statistics.
# The two functions below are copied verbatim from 02_graph_construction.ipynb
# so the Dutch graph is comparable to English BY CONSTRUCTION (same code path).

import networkx as nx

def clean_swow(df: pd.DataFrame, min_r123: int = 2) -> pd.DataFrame:
    """Five preprocessing rules — identical to Phase 1."""
    df = df.copy()
    df = df.dropna(subset=['cue', 'response'])
    df['cue'] = df['cue'].str.lower()              # Rule 1
    df['response'] = df['response'].str.lower()    # Rule 1
    df = df[df['response'].str.len() > 1]          # Rule 2: drop single-char (kills 'x')
    df = df[~df['response'].str.contains(r'\s', regex=True)]  # Rule 3: multi-word
    df = df[df['response'].str.isalpha()]          # Rule 4: alphabetic only (keeps ë, ï)
    df = df[df['R123'] >= min_r123]                # Rule 5: min participant count
    return df.reset_index(drop=True)

def aggregate_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Sum counts for (cue,response) pairs that collapsed under lowercasing —
    identical to Phase 1; this is the step that caught the 130-row bug in EN."""
    df = df.copy()
    agg = (df.groupby(['cue', 'response'], as_index=False)
             .agg({'R123': 'sum', 'N': 'max'}))
    agg['R123.Strength'] = agg['R123'] / agg['N']
    return agg

# --- Run the identical pipeline ---------------------------------------------
n_raw = len(nl_strength)
nl_clean = clean_swow(nl_strength)
n_clean = len(nl_clean)

# Report how many duplicates the lowercasing collapsed (the EN bug-check)
nl_clean_agg = aggregate_duplicates(nl_clean)
n_merged = len(nl_clean) - len(nl_clean_agg)

print(f"Raw strength rows:        {n_raw:,}")
print(f"After 5 cleaning rules:   {n_clean:,}  (retention {100*n_clean/n_raw:.1f}%)")
print(f"Duplicate pairs merged:   {n_merged:,}   <-- watch this, EN had 130")

# --- Build the directed graph (same call as Phase 1) ------------------------
G_nl = nx.from_pandas_edgelist(
    nl_clean_agg, source='cue', target='response',
    edge_attr='R123.Strength', create_using=nx.DiGraph()
)

# --- Phase-1-equivalent statistics ------------------------------------------
n_nodes = G_nl.number_of_nodes()
n_edges = G_nl.number_of_edges()
cues = {n for n in G_nl.nodes() if G_nl.out_degree(n) > 0}
response_only = n_nodes - len(cues)
largest_wcc = max(nx.weakly_connected_components(G_nl), key=len)

print(f"\n=== Dutch graph (compare to EN: 24,657 nodes / 279,410 edges) ===")
print(f"  Nodes:              {n_nodes:,}")
print(f"  Edges:              {n_edges:,}")
print(f"  Density:            {nx.density(G_nl):.6e}   (EN: 4.60e-4)")
print(f"  Cue nodes (out>0):  {len(cues):,}")
print(f"  Response-only:      {response_only:,}  ({100*response_only/n_nodes:.1f}%)")
print(f"  Largest WCC:        {len(largest_wcc):,} ({100*len(largest_wcc)/n_nodes:.1f}%)")

# --- Spot check: 'water' from the graph, like EN's 'ocean' check ------------
print(f"\n=== 'water' neighbors from graph (top 6) — compare to raw table above ===")
nb = sorted(((t, d['R123.Strength']) for t, d in G_nl['water'].items()),
            key=lambda x: x[1], reverse=True)[:6]
for t, w in nb:
    print(f"  water → {t:<12} {w:.4f}")

Raw strength rows:        1,497,336
After 5 cleaning rules:   423,578  (retention 28.3%)
Duplicate pairs merged:   0   <-- watch this, EN had 130

=== Dutch graph (compare to EN: 24,657 nodes / 279,410 edges) ===
  Nodes:              35,927
  Edges:              423,578
  Density:            3.281736e-04   (EN: 4.60e-4)
  Cue nodes (out>0):  12,571
  Response-only:      23,356  (65.0%)
  Largest WCC:        35,927 (100.0%)

=== 'water' neighbors from graph (top 6) — compare to raw table above ===
  water → drinken      0.3100
  water → dorst        0.2700
  water → zee          0.2700
  water → zwemmen      0.2200
  water → nat          0.1500
  water → blauw        0.1400


In [9]:
# Cell 5 — Quantify the response-only population BEFORE building cue-cores.
# WHY: the EN→NL response-only jump (35%→65%) is the biggest structural
# difference on the table. Option C drops these nodes for the primary
# comparison, so we need to know what we're dropping: mostly degree-1 leaves
# (the "forced third response" artifact, safe to drop) or higher-degree
# response-only words (real structural signal, costly to drop)?
#
# Requires G_nl (this notebook) and the English graph from Phase 1.

import pickle
import networkx as nx

# Load the English graph built in Phase 1.
with open(ROOT / "data/processed/swow_en_graph.pkl", "rb") as f:
    G_en = pickle.load(f)

def response_only_profile(G, name):
    """Characterize nodes with no outgoing edges (response-only)."""
    n_total = G.number_of_nodes()
    resp_only = [n for n in G.nodes() if G.out_degree(n) == 0]
    n_resp = len(resp_only)

    # In-degree distribution among response-only nodes.
    indeg = [G.in_degree(n) for n in resp_only]
    deg1 = sum(1 for d in indeg if d == 1)          # pure leaves
    deg2 = sum(1 for d in indeg if d == 2)
    deg3plus = sum(1 for d in indeg if d >= 3)
    deg10plus = sum(1 for d in indeg if d >= 10)    # structurally notable

    print(f"=== {name} ===")
    print(f"  Total nodes:            {n_total:,}")
    print(f"  Response-only nodes:    {n_resp:,} ({100*n_resp/n_total:.1f}%)")
    print(f"  ...of which in-deg = 1: {deg1:,} ({100*deg1/n_resp:.1f}% of resp-only)")
    print(f"  ...in-deg = 2:          {deg2:,} ({100*deg2/n_resp:.1f}%)")
    print(f"  ...in-deg >= 3:         {deg3plus:,} ({100*deg3plus/n_resp:.1f}%)")
    print(f"  ...in-deg >= 10:        {deg10plus:,} ({100*deg10plus/n_resp:.1f}%)")
    print(f"  Max in-degree among response-only: {max(indeg) if indeg else 0}")

    # What fraction of ALL edges touch a response-only node as target?
    edges_to_resp = sum(1 for _, v in G.edges() if G.out_degree(v) == 0)
    print(f"  Edges pointing INTO response-only nodes: {edges_to_resp:,} "
          f"({100*edges_to_resp/G.number_of_edges():.1f}% of all edges)")
    print()
    return resp_only

resp_en = response_only_profile(G_en, "ENGLISH (full graph)")
resp_nl = response_only_profile(G_nl, "DUTCH (full graph)")

=== ENGLISH (full graph) ===
  Total nodes:            24,657
  Response-only nodes:    15,998 (64.9%)
  ...of which in-deg = 1: 7,508 (46.9% of resp-only)
  ...in-deg = 2:          2,858 (17.9%)
  ...in-deg >= 3:         5,632 (35.2%)
  ...in-deg >= 10:        1,547 (9.7%)
  Max in-degree among response-only: 862
  Edges pointing INTO response-only nodes: 96,315 (34.5% of all edges)

=== DUTCH (full graph) ===
  Total nodes:            35,927
  Response-only nodes:    23,356 (65.0%)
  ...of which in-deg = 1: 12,843 (55.0% of resp-only)
  ...in-deg = 2:          4,348 (18.6%)
  ...in-deg >= 3:         6,165 (26.4%)
  ...in-deg >= 10:        562 (2.4%)
  Max in-degree among response-only: 169
  Edges pointing INTO response-only nodes: 55,531 (13.1% of all edges)



In [10]:
# Cell 6 — Build the cue-core of each graph: the subgraph induced on cue nodes
# (nodes with at least one outgoing edge). This is the PRIMARY comparison level
# for Option C — it removes the response-only leaf periphery so cross-linguistic
# differences aren't dominated by the protocol-driven leaf asymmetry.
#
# NOTE: an edge cue_A -> cue_B survives only if BOTH endpoints are cues.
# An edge cue_A -> response_only_word is dropped (target isn't a cue).
# So the core is "associations among stimulus words only."

import networkx as nx

def build_cue_core(G, name):
    cues = [n for n in G.nodes() if G.out_degree(n) > 0]
    core = G.subgraph(cues).copy()   # induced subgraph: keeps edges with both ends in `cues`

    n_full_nodes, n_full_edges = G.number_of_nodes(), G.number_of_edges()
    n_core_nodes, n_core_edges = core.number_of_nodes(), core.number_of_edges()

    # Connectivity of the core (it may fragment once leaves are gone)
    wccs = list(nx.weakly_connected_components(core))
    largest = max(wccs, key=len)

    print(f"=== {name} cue-core ===")
    print(f"  Nodes: {n_core_nodes:,}  (was {n_full_nodes:,} full — kept {100*n_core_nodes/n_full_nodes:.1f}%)")
    print(f"  Edges: {n_core_edges:,}  (was {n_full_edges:,} full — kept {100*n_core_edges/n_full_edges:.1f}%)")
    print(f"  Weakly connected components: {len(wccs):,}")
    print(f"  Largest WCC: {len(largest):,} ({100*len(largest)/n_core_nodes:.1f}% of core)")
    print(f"  Mean out-degree: {n_core_edges/n_core_nodes:.1f}")
    print(f"  Density: {nx.density(core):.6e}")
    print()
    return core

core_en = build_cue_core(G_en, "ENGLISH")
core_nl = build_cue_core(G_nl, "DUTCH")

=== ENGLISH cue-core ===
  Nodes: 8,659  (was 24,657 full — kept 35.1%)
  Edges: 183,095  (was 279,410 full — kept 65.5%)
  Weakly connected components: 1
  Largest WCC: 8,659 (100.0% of core)
  Mean out-degree: 21.1
  Density: 2.442256e-03

=== DUTCH cue-core ===
  Nodes: 12,571  (was 35,927 full — kept 35.0%)
  Edges: 368,047  (was 423,578 full — kept 86.9%)
  Weakly connected components: 1
  Largest WCC: 12,571 (100.0% of core)
  Mean out-degree: 29.3
  Density: 2.329154e-03



In [11]:
# Cell 7 — Normalization scaffolding: tools to compare size- and degree-
# mismatched graphs fairly. Built BEFORE any topology metric so every later
# metric is reported against an appropriate null, not raw.
#
# Two distinct confounds, two distinct tools:
#   (A) matched subsampling   -> controls for raw SIZE (node count)
#   (B) configuration model   -> controls for DEGREE SEQUENCE
# We validate (A) on density, a quantity we already understand from Cell 6.

import networkx as nx
import numpy as np

RNG = np.random.default_rng(42)  # reproducible

def subsample_metric(G, target_n, metric_fn, n_reps=30, rng=RNG):
    """Draw n_reps random target_n-node induced subgraphs of G; return the
    distribution of metric_fn over them. Controls for raw size."""
    nodes = list(G.nodes())
    vals = []
    for _ in range(n_reps):
        chosen = rng.choice(len(nodes), size=target_n, replace=False)
        sub = G.subgraph([nodes[i] for i in chosen]).copy()
        vals.append(metric_fn(sub))
    return np.array(vals)

def config_null_metric(G, metric_fn, n_reps=30, rng=RNG):
    """Generate n_reps degree-preserving random graphs (directed configuration
    model: preserves in- and out-degree sequences), strip self-loops/parallels,
    return distribution of metric_fn. Controls for degree sequence.
    NOTE: operates on UNWEIGHTED directed structure; weights are NOT preserved."""
    din  = [d for _, d in G.in_degree()]
    dout = [d for _, d in G.out_degree()]
    vals = []
    for _ in range(n_reps):
        seed = int(rng.integers(1_000_000))
        rand = nx.directed_configuration_model(din, dout, seed=seed)
        rand = nx.DiGraph(rand)          # collapse parallel edges
        rand.remove_edges_from(nx.selfloop_edges(rand))  # drop self-loops
        vals.append(metric_fn(rand))
    return np.array(vals)

# --- VALIDATION on density (we already understand this from Cell 6) ---------
# Subsample the larger Dutch core down to English's size and check density.
# If the size confound explains the density picture, subsampled-Dutch density
# should behave predictably relative to the raw numbers we already have.
en_n = core_en.number_of_nodes()   # 8,659

nl_sub_density = subsample_metric(core_nl, en_n, nx.density, n_reps=30)

print("=== Validation: density under matched subsampling ===")
print(f"English core density (actual):      {nx.density(core_en):.6e}")
print(f"Dutch core density (actual, full):  {nx.density(core_nl):.6e}")
print(f"Dutch subsampled to EN size (n={en_n}):")
print(f"  mean:  {nl_sub_density.mean():.6e}")
print(f"  std:   {nl_sub_density.std():.6e}")
print(f"  range: [{nl_sub_density.min():.6e}, {nl_sub_density.max():.6e}]")

=== Validation: density under matched subsampling ===
English core density (actual):      2.442256e-03
Dutch core density (actual, full):  2.329154e-03
Dutch subsampled to EN size (n=8659):
  mean:  2.324185e-03
  std:   4.616387e-05
  range: [2.219392e-03, 2.425703e-03]


In [12]:
# Cell 8 — Clustering coefficient, normalized against both nulls.
# Density told us HOW MANY links exist; clustering tells us how they're ARRANGED
# (do neighbors link to each other -> local triangles).
# We report raw clustering AND clustering vs (a) size-matched subsample,
# (b) degree-preserving config null. A difference must survive both to count.

import networkx as nx
import numpy as np

# Clustering on the undirected projection (standard for this metric).
clust = lambda G: nx.average_clustering(G.to_undirected())

en_clust = clust(core_en)
nl_clust = clust(core_nl)

print("=== Raw average clustering ===")
print(f"  English core: {en_clust:.4f}")
print(f"  Dutch core:   {nl_clust:.4f}\n")

# (a) Size control: Dutch subsampled to English's size.
en_n = core_en.number_of_nodes()
nl_sub_clust = subsample_metric(core_nl, en_n, clust, n_reps=30)
print("=== Size control: Dutch subsampled to EN size ===")
print(f"  Dutch subsampled clustering: {nl_sub_clust.mean():.4f} "
      f"± {nl_sub_clust.std():.4f}  range [{nl_sub_clust.min():.4f}, {nl_sub_clust.max():.4f}]")
print(f"  English actual:              {en_clust:.4f}\n")

# (b) Degree control: how much higher is real clustering than a degree-matched
# random graph? Ratio > 1 means real structure clusters more than degree alone explains.
en_null = config_null_metric(core_en, clust, n_reps=20)
nl_null = config_null_metric(core_nl, clust, n_reps=20)
print("=== Degree control: clustering vs degree-preserving null ===")
print(f"  English: real {en_clust:.4f} / null {en_null.mean():.4f} = ratio {en_clust/en_null.mean():.1f}x")
print(f"  Dutch:   real {nl_clust:.4f} / null {nl_null.mean():.4f} = ratio {nl_clust/nl_null.mean():.1f}x")

=== Raw average clustering ===
  English core: 0.1762
  Dutch core:   0.2014

=== Size control: Dutch subsampled to EN size ===
  Dutch subsampled clustering: 0.2011 ± 0.0027  range [0.1946, 0.2058]
  English actual:              0.1762

=== Degree control: clustering vs degree-preserving null ===
  English: real 0.1762 / null 0.0188 = ratio 9.4x
  Dutch:   real 0.2014 / null 0.0250 = ratio 8.0x


In [13]:
# Cell 9 — PageRank hub structure. Who are the most central words in each
# language, and is the *pattern* of centrality similar?
# PageRank = a word is important if many important words point to it.
# Uses edge weights (R123.Strength) — strong associations count more.

import networkx as nx
import numpy as np

pr_en = nx.pagerank(core_en, weight='R123.Strength')
pr_nl = nx.pagerank(core_nl, weight='R123.Strength')

def top_hubs(pr, n=20):
    return sorted(pr.items(), key=lambda x: x[1], reverse=True)[:n]

print("=== Top 20 hubs: ENGLISH ===")
for w, s in top_hubs(pr_en):
    print(f"  {w:<15} {s:.5f}")

print("\n=== Top 20 hubs: DUTCH ===")
for w, s in top_hubs(pr_nl):
    print(f"  {w:<15} {s:.5f}")

# --- Is centrality concentrated in a few hubs, or spread out? ---------------
# Gini-like check: what share of total PageRank do the top 1% of words hold?
def top_share(pr, frac=0.01):
    vals = np.sort(np.array(list(pr.values())))[::-1]
    k = max(1, int(len(vals) * frac))
    return vals[:k].sum()

print("\n=== Centrality concentration (share held by top 1% of words) ===")
print(f"  English: {top_share(pr_en):.3f}")
print(f"  Dutch:   {top_share(pr_nl):.3f}")

=== Top 20 hubs: ENGLISH ===
  water           0.00745
  money           0.00646
  white           0.00606
  love            0.00555
  blue            0.00548
  red             0.00510
  man             0.00475
  small           0.00407
  music           0.00404
  time            0.00374
  woman           0.00374
  work            0.00374
  car             0.00372
  child           0.00366
  sky             0.00351
  black           0.00345
  sad             0.00335
  tree            0.00330
  sun             0.00324
  light           0.00313

=== Top 20 hubs: DUTCH ===
  zon             0.00764
  water           0.00739
  warm            0.00682
  eten            0.00677
  geld            0.00623
  zee             0.00532
  lekker          0.00511
  groen           0.00500
  rood            0.00477
  pijn            0.00468
  mooi            0.00466
  leuk            0.00405
  school          0.00398
  vakantie        0.00386
  auto            0.00379
  wit             0.00377
  zomer

In [14]:
# Cell 10 — Size-validate the hub-concentration finding, then path length.
# Part 1: does Dutch's higher centrality concentration survive size-matching?
# Part 2: characteristic path length — but per Phase 2, this is a PURELY
# STRUCTURAL observation with NO validated cognitive meaning attached.

import networkx as nx
import numpy as np

# --- Part 1: concentration under size control -------------------------------
def top_share_pr(G, frac=0.01):
    pr = nx.pagerank(G, weight='R123.Strength')
    vals = np.sort(np.array(list(pr.values())))[::-1]
    k = max(1, int(len(vals) * frac))
    return vals[:k].sum()

en_n = core_en.number_of_nodes()
en_conc = top_share_pr(core_en)
nl_sub_conc = subsample_metric(core_nl, en_n, top_share_pr, n_reps=20)

print("=== Hub concentration (top 1% share), size-controlled ===")
print(f"  English actual:             {en_conc:.3f}")
print(f"  Dutch subsampled to EN size: {nl_sub_conc.mean():.3f} "
      f"± {nl_sub_conc.std():.3f}  range [{nl_sub_conc.min():.3f}, {nl_sub_conc.max():.3f}]")
print()

# --- Part 2: characteristic path length -------------------------------------
# Average shortest-path length on the largest strongly-connected component
# (directed: we need every node reachable from every other for this to be defined).
# STRUCTURAL ONLY — Phase 2 showed path length carries no behavioral signal.
def char_path_length(G):
    scc = max(nx.strongly_connected_components(G), key=len)
    sub = G.subgraph(scc)
    return nx.average_shortest_path_length(sub), len(scc)

en_apl, en_scc = char_path_length(core_en)
nl_apl, nl_scc = char_path_length(core_nl)

print("=== Characteristic path length (largest SCC, unweighted, directed) ===")
print(f"  English: {en_apl:.3f} hops  (SCC = {en_scc:,} of {core_en.number_of_nodes():,} nodes)")
print(f"  Dutch:   {nl_apl:.3f} hops  (SCC = {nl_scc:,} of {core_nl.number_of_nodes():,} nodes)")

=== Hub concentration (top 1% share), size-controlled ===
  English actual:             0.216
  Dutch subsampled to EN size: 0.296 ± 0.008  range [0.278, 0.312]

=== Characteristic path length (largest SCC, unweighted, directed) ===
  English: 3.814 hops  (SCC = 8,139 of 8,659 nodes)
  Dutch:   3.791 hops  (SCC = 11,661 of 12,571 nodes)


In [15]:
# Cell 11 — Community structure via modularity.
# Do words split into tight thematic groups (food, colors, emotions...)?
# Modularity Q: how cleanly the network partitions into communities.
#   higher Q = sharper, more separable communities.
# We use Louvain (greedy, standard). Run on undirected projection.
# Size/degree confounds apply, so we ALSO compare each Q to its degree-null.

import networkx as nx
import numpy as np

def modularity_Q(G):
    UG = G.to_undirected()
    comms = nx.community.louvain_communities(UG, weight='R123.Strength', seed=42)
    Q = nx.community.modularity(UG, comms, weight='R123.Strength')
    return Q, len(comms), sorted((len(c) for c in comms), reverse=True)

en_Q, en_k, en_sizes = modularity_Q(core_en)
nl_Q, nl_k, nl_sizes = modularity_Q(core_nl)

print("=== Modularity (Louvain, weighted) ===")
print(f"  English: Q = {en_Q:.3f}, {en_k} communities, "
      f"largest 5 sizes: {en_sizes[:5]}")
print(f"  Dutch:   Q = {nl_Q:.3f}, {nl_k} communities, "
      f"largest 5 sizes: {nl_sizes[:5]}\n")

# Degree control: real Q vs Q of degree-preserving null (unweighted).
def mod_Q_unweighted(G):
    UG = G.to_undirected()
    comms = nx.community.louvain_communities(UG, seed=42)
    return nx.community.modularity(UG, comms)

en_Q_real = mod_Q_unweighted(core_en)
nl_Q_real = mod_Q_unweighted(core_nl)
en_Q_null = config_null_metric(core_en, mod_Q_unweighted, n_reps=10)
nl_Q_null = config_null_metric(core_nl, mod_Q_unweighted, n_reps=10)

print("=== Modularity vs degree-preserving null (unweighted) ===")
print(f"  English: real {en_Q_real:.3f} / null {en_Q_null.mean():.3f}")
print(f"  Dutch:   real {nl_Q_real:.3f} / null {nl_Q_null.mean():.3f}")

=== Modularity (Louvain, weighted) ===
  English: Q = 0.495, 21 communities, largest 5 sizes: [1055, 626, 585, 566, 534]
  Dutch:   Q = 0.482, 18 communities, largest 5 sizes: [1322, 1220, 1165, 1077, 809]

=== Modularity vs degree-preserving null (unweighted) ===
  English: real 0.374 / null 0.130
  Dutch:   real 0.380 / null 0.109


In [16]:
# Cell 12 — Synthesis: assemble all RQ2 metrics into one comparison table.
# Each row: English value, Dutch value, whether the difference survives
# size-control, and the verdict. This is the backbone of the results section.

import pandas as pd

# Values established across cells 6-11 (re-stating, not recomputing the heavy ones).
rows = [
    # metric,            EN,        NL,        size-controlled?,           verdict
    ("Nodes (core)",     "8,659",   "12,571",  "n/a (raw size)",           "Dutch larger (dataset)"),
    ("Edges (core)",     "183,095", "368,047", "n/a (raw size)",           "Dutch larger (dataset)"),
    ("Density",          "2.44e-3", "2.33e-3", "EN just above NL range",   "~equal"),
    ("Mean out-degree",  "21.1",    "29.3",    "confounded by size",       "higher raw, see density"),
    ("Char. path length","3.81",    "3.79",    "n/a (both small-world)",   "~equal (structural only)"),
    ("Clustering",       "0.176",   "0.201",   "YES — EN below NL range",  "Dutch higher (robust)"),
    ("Hub concentration","0.216",   "0.297",   "YES — EN below NL range",  "Dutch higher (robust)"),
    ("Modularity Q",     "0.495",   "0.482",   "both >> degree-null",      "~equal"),
    ("# communities",    "21",      "18",      "size-sensitive",           "Dutch fewer"),
    ("Community sizes",  "evener",  "chunkier","size-sensitive",           "Dutch coarser-grained"),
]

summary = pd.DataFrame(rows, columns=["Metric", "English", "Dutch",
                                       "Size-controlled?", "Verdict"])
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)
print(summary.to_string(index=False))

# Save for the write-up.
out = ROOT / "data/processed/phase3_comparison_summary.csv"
summary.to_csv(out, index=False)
print(f"\nSaved: {out}")

           Metric English    Dutch        Size-controlled?                  Verdict
     Nodes (core)   8,659   12,571          n/a (raw size)   Dutch larger (dataset)
     Edges (core) 183,095  368,047          n/a (raw size)   Dutch larger (dataset)
          Density 2.44e-3  2.33e-3  EN just above NL range                   ~equal
  Mean out-degree    21.1     29.3      confounded by size  higher raw, see density
Char. path length    3.81     3.79  n/a (both small-world) ~equal (structural only)
       Clustering   0.176    0.201 YES — EN below NL range    Dutch higher (robust)
Hub concentration   0.216    0.297 YES — EN below NL range    Dutch higher (robust)
     Modularity Q   0.495    0.482     both >> degree-null                   ~equal
    # communities      21       18          size-sensitive              Dutch fewer
  Community sizes  evener chunkier          size-sensitive    Dutch coarser-grained

Saved: /Users/mac/Desktop/semantic_graph_project/data/processed/phase3_comp

In [17]:
# Cell 13 — Protocol robustness check.
# Dutch forced 3 responses; English allowed early stop. If our clustering/
# concentration differences are a forced-response artifact, they should SHRINK
# when we rebuild Dutch from asso1 only (mimicking a single-response protocol).
# We reuse the EXACT same pipeline, just restricted to the first slot.

import networkx as nx
import numpy as np

# --- Rebuild Dutch strength from asso1 ONLY ---------------------------------
a1 = nl[["recodedPP_ID", "cue", "asso1"]].copy()
a1["cue"] = a1["cue"].str.lower().str.strip()
a1["response"] = a1["asso1"].str.lower().str.strip()
a1 = a1[a1["response"] != ""]

N1 = a1.groupby("cue")["recodedPP_ID"].nunique()          # N per cue (asso1)
R1 = (a1.drop_duplicates(["cue", "response", "recodedPP_ID"])
        .groupby(["cue", "response"]).size().rename("R123").reset_index())
nl1 = R1.merge(N1.rename("N"), on="cue").assign(
    **{"R123.Strength": lambda d: d["R123"] / d["N"]})

# Same cleaning + aggregation + graph build, then take cue-core.
nl1_clean = aggregate_duplicates(clean_swow(nl1))
G_nl1 = nx.from_pandas_edgelist(nl1_clean, "cue", "response",
                                edge_attr="R123.Strength", create_using=nx.DiGraph())
core_nl1 = G_nl1.subgraph([n for n in G_nl1 if G_nl1.out_degree(n) > 0]).copy()

print(f"Dutch asso1-only core: {core_nl1.number_of_nodes():,} nodes, "
      f"{core_nl1.number_of_edges():,} edges, mean out-deg "
      f"{core_nl1.number_of_edges()/core_nl1.number_of_nodes():.1f}")

# --- Recompute the two contested metrics ------------------------------------
clust = lambda G: nx.average_clustering(G.to_undirected())
def top_share_pr(G, frac=0.01):
    pr = nx.pagerank(G, weight='R123.Strength')
    v = np.sort(np.array(list(pr.values())))[::-1]
    return v[:max(1, int(len(v)*frac))].sum()

print("\n=== Does the difference survive single-response Dutch? ===")
print(f"{'metric':<20}{'English':>10}{'Dutch(3)':>12}{'Dutch(asso1)':>14}")
print(f"{'clustering':<20}{0.176:>10.3f}{0.201:>12.3f}{clust(core_nl1):>14.3f}")
print(f"{'hub concentration':<20}{0.216:>10.3f}{0.297:>12.3f}{top_share_pr(core_nl1):>14.3f}")

Dutch asso1-only core: 12,571 nodes, 127,031 edges, mean out-deg 10.1

=== Does the difference survive single-response Dutch? ===
metric                 English    Dutch(3)  Dutch(asso1)
clustering               0.176       0.201         0.228
hub concentration        0.216       0.297         0.347
